# Ho gK-cell tuning — rasters while you tune

Derived from `neuron_network_simulation.ipynb`, adapted for the **Ho 2025 gK cells**
(branch `sim/ho-gk-cells`). The intrinsic model is now the Ho conductance set; the
4-AP knob is the delayed rectifier **`gK`** (`gK_exc` = PY, `gK_inh` = FS). `sahp_*`
and `adapt` are **inert** here (Ho cells have no sAHP — `ikCa` supplies adaptation),
so they are omitted from the build config.

**How to tune:** run the setup cells once, then edit the `run_and_raster(...)` call in
the **Tune here** cell and re-run it. Each run prints a one-line summary and shows a
cluster-sorted raster (blue = excitatory, red = inhibitory; bottom panel = fraction
active in 10 ms bins). Iterate on `TOPO_FAST` (small, ~seconds per run); validate on
`TOPO_FULL` at the end.

**Goal:** turn the untuned **asynchronous tonic firing** into **discrete synchronized
bursts** with quiet inter-burst intervals (and a finite `[K+]o`).

> Kernel: **Python 3.9 (NEURON)**. If `import` of `neuron`/mechanisms fails, the kernel
> isn't the NEURON one — pick `Python 3.9 (NEURON)` from the kernel menu.

In [ ]:
%matplotlib inline

In [ ]:
import os, sys, gc, time
import numpy as np
import matplotlib.pyplot as plt
from neuron import h

REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "inference")):
    if p not in sys.path:
        sys.path.insert(0, p)

from neuron_simulation import (
    topology, build_network, run_simulation, states, analysis, plotting,
)
from neuron_simulation.topology import NeuronWeightParameters
from neuron_simulation import neurons_ho

neurons_ho.load_mechanisms()       # loads neuron_simulation/nrnmech.dll (Ho + kdyn + syn)

# --- synaptic weight ranges (uS); part of the topology draw ---
wp = NeuronWeightParameters()
wp.within_exc_range  = (0.0010, 0.0022)
wp.between_exc_range = (0.0008, 0.0016)
wp.within_inh_range  = (0.0025, 0.0055)
wp.between_inh_range = (0.0020, 0.0040)
wp.use_lognormal     = True
wp.lognormal_sigma   = 0.5

# --- topology config (from the original notebook: 50-cluster, cell-type-specific) ---
TOPO_CFG = dict(
    num_clusters=50, neurons_per_cluster_range=(4, 40), inhibitory_probability=0.2,
    cluster_radius=1.0, space_size=15.0, seed=1,
    decay_sigma=3.0, max_connection_distance=6.0,
    cell_type_specific=True,
    p_ee_within=0.2, p_ee_between=0.1, p_ei_within=0.20, p_ei_between=0.08,
    p_ie_within=0.40, p_ii_within=0.50,
    within_cluster_prob=0.25, between_cluster_prob=0.06,   # ignored (cell_type_specific=True)
    ln_sigma=0.5, target_density=None, weight_params=wp,
)

# --- Ho build kwargs (starting point; sahp_*/adapt are INERT with Ho cells -> omitted) ---
BASE_BUILD = dict(
    celsius=6.3,
    synapse_model="ampa_nmda", exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
    inh_tau=6.0, e_inh=-75.0,
    exc_weight_scale=2.0, inh_weight_scale=2.5,
    depression=True, depression_d=0.2, tau_d=500.0,
    noise_rate=5.0, noise_weight=0.004, noise_tau=3.0,
    syn_delay=1.5, delay_per_distance=2.0,
)
SIM = dict(dt=0.025, discard_transient_ms=500.0)  # dt<=0.025: Ho cells are unstable at dt=0.05

# fast (small) topology for quick tuning; full for validation
TOPO_FAST = topology.build_topology_lognormal(**{**TOPO_CFG, "num_clusters": 8})
TOPO_FULL = topology.build_topology_lognormal(**TOPO_CFG)
print(f"TOPO_FAST: {TOPO_FAST['n_neurons']} neurons | TOPO_FULL: {TOPO_FULL['n_neurons']} neurons")

## Topology — fast tuning network

Four-panel overview + printed connection stats (density, within/between rates, hubs) for `TOPO_FAST`.

In [ ]:
stats, fig = plotting.topology_report(TOPO_FAST)
plt.show()

## The tuning helper

`run_and_raster(...)` builds a fresh network with your overrides, runs it, prints a
summary, and shows the raster. The network is **not** retained, so NEURON state does
not accumulate across reruns. Any `build_network` kwarg can be passed as an override.

In [ ]:
def run_and_raster(topo=TOPO_FAST, gK_exc=15.0, gK_inh=10.0, tau_k=200.0,
                   duration=2000.0, noise_seed=1, show_ko=True, randomize=False,
                   title=None, **build_overrides):
    """Build -> run -> show a cluster-sorted raster. Edit kwargs and re-run.

    build_overrides: any build_network kwarg, e.g. exc_weight_scale, inh_weight_scale,
    noise_rate, noise_weight, nmda_ratio, tau_nmda, depression_d, ...

    The network is built locally and NOT returned, so it is freed when this call
    returns -- NEURON state does not accumulate across reruns (no manual
    delete_section, which segfaults on live references).
    """
    gc.collect()   # free any prior network before building the next one
    N = topo["n_neurons"]
    bk = dict(BASE_BUILD)
    bk.update(gK_exc=gK_exc, gK_inh=gK_inh, tau_k=tau_k, noise_seed=noise_seed)
    bk.update(build_overrides)
    _t0 = time.time()
    net = build_network(topo, **bk)
    spikes, _v, ko = run_simulation(net, duration=duration, record_ko=show_ko,
                                    progress_every_ms=None, **SIM)
    _elapsed = time.time() - _t0
    lb = analysis.detect_network_bursts(spikes, N, duration,
                                        participation_threshold=0.35, burn_in_ms=0.0)
    lbs = analysis.burst_statistics(lb, duration, burn_in_ms=0.0)
    rate = analysis.firing_rate_summary(spikes, duration, burn_in_ms=0.0)["mean_rate_hz"]
    ko_ok = (show_ko and ko is not None
             and np.isfinite(np.asarray(ko["mean_ko"], float)).all())
    ko_txt = (f" | [K+]o {float(ko['mean_ko'].min()):.1f}-{float(ko['mean_ko'].max()):.1f} mM"
              if ko_ok else " | [K+]o n/a")
    print(f"N={N} | {duration:.0f}ms in {_elapsed:.0f}s | rate={rate:.1f}Hz | "
          f"loose bursts {lbs['n_bursts']} ({lbs['burst_rate_hz']:.2f}Hz, "
          f"partic {lbs['mean_participation']:.2f}){ko_txt}")
    ttl = title or (f"gK=({gK_exc},{gK_inh}) tau_k={tau_k} | exc x{bk['exc_weight_scale']} "
                    f"inh x{bk['inh_weight_scale']} | noise {bk['noise_rate']}Hz/{bk['noise_weight']}")
    if ko_ok:
        fig = plotting.plot_raster_with_ko(spikes, N, duration, ko,
              is_inhibitory=topo["neuron_is_inhibitory"],
              cluster_assignments=topo["cluster_assignments"],
              burn_in_ms=0.0, title=ttl, randomize_rows=randomize)
    else:
        fig = plotting.plot_raster(spikes, N, duration,
              is_inhibitory=topo["neuron_is_inhibitory"],
              cluster_assignments=topo["cluster_assignments"],
              burn_in_ms=0.0, title=ttl, randomize_rows=randomize)
    plt.show()
    cfg = dict(net.config)           # plain-dict copy (no live NEURON refs)
    return dict(spikes=spikes, ko=ko, burst_stats=lbs, rate=rate, fig=fig, config=cfg)

## Tune here — edit the call and re-run

Edit the arguments below and re-run **this cell** to see the raster. Start on `TOPO_FAST`
with a short `duration`; see the tuning guide further down for which way to push each knob.

In [ ]:
# EDIT these, then re-run this cell.  Each run: summary line + raster.
res = run_and_raster(
    topo=TOPO_FAST,          # TOPO_FAST (~seconds) while tuning; TOPO_FULL to validate
    duration=2000,           # ms; shorter = faster iteration

    # ---- 4-AP / seizure knobs ----
    gK_exc=15.0,             # PY delayed rectifier: 15 = normal, ~0.3 = 4-AP cascade
    gK_inh=10.0,             # FS delayed rectifier
    tau_k=200.0,             # K+ clearance: 200 = normal, larger = seizure ([K+]o accumulates)

    # ---- E/I balance (main knobs to calm the runaway activity) ----
    exc_weight_scale=2.0,    # recurrent excitation gain -> LOWER to calm
    inh_weight_scale=2.5,    # recurrent inhibition gain -> RAISE to calm

    # ---- background drive ----
    noise_rate=5.0,          # Hz per cell -> LOWER for sparser firing
    noise_weight=0.004,      # uS

    # ---- synaptic kinetics ----
    nmda_ratio=3.0, tau_nmda=350.0,
    depression_d=0.2,        # short-term depression -> RAISE to help bursts terminate
    # randomize=True,        # uncomment: check burst synchrony survives a row-shuffle
)

## Compare normal vs 4-AP (gK down)

Same network and tuning; only `gK_exc` differs (15 → 0.3). Expect the 4-AP run to fire slower with broader spikes.

In [ ]:
# same network + tuning, only gK differs
_ = run_and_raster(topo=TOPO_FAST, duration=2000, gK_exc=15.0, gK_inh=10.0, title='normal gK')
_ = run_and_raster(topo=TOPO_FAST, duration=2000, gK_exc=0.3,  gK_inh=10.0, title='4-AP: PY gK=0.3')

## Tuning guide

The untuned network fires **asynchronously and tonically** (weights were set for `hh`+`kA`).
To recover **discrete synchronized bursts**, push these knobs:

| Knob | What it is | For bursting |
|---|---|---|
| `exc_weight_scale` | recurrent excitation gain | **lower** (try 0.5–1.5) to stop runaway firing |
| `inh_weight_scale` | recurrent inhibition gain | **raise** to tighten synchrony / gate bursts |
| `noise_rate`, `noise_weight` | background drive | **lower** so cells are quiet between bursts |
| `depression_d` (`tau_d`) | short-term synaptic depression | **raise** so bursts self-terminate |
| `nmda_ratio`, `tau_nmda` | slow NMDA depolarization | moderate NMDA promotes synchronized bursts; too much → sustained |
| `tau_k` | `[K+]o` clearance (seizure knob) | 200 = normal; **raise** for the K+-accumulation seizure |
| `gK_exc` | PY delayed rectifier (4-AP knob) | 15 normal → ~0.3 for the 4-AP cascade (broad spikes) |
| `wp.*` ranges | per-edge weight draw | **rewires** — re-run the setup cell after changing |

**Read-outs** (printed each run): `rate` (aim well below the ~30 Hz untuned tonic level),
`loose bursts` (participation ≥ 35 %; want a few Hz at high participation), and `[K+]o`
(becomes finite once activity is tamed — `nan` means a numeric blow-up from over-excitation).

Anything in `wp` / `TOPO_CFG` **rewires** the network — re-run the **setup** cell to rebuild
`TOPO_FAST` / `TOPO_FULL` before the next `run_and_raster`.

## Validate on the full 50-cluster network

**Slow** (~minutes on ~900 cells). Run once the fast-network dynamics look right.

In [ ]:
# SLOW (~minutes). Validate your tuned knobs on the full network.
res_full = run_and_raster(topo=TOPO_FULL, duration=3000, gK_exc=15.0, gK_inh=10.0,
                          exc_weight_scale=2.0, inh_weight_scale=2.5, noise_rate=5.0)